<h1>مجموعهٔ تست‌ها (<code>GIVEN</code>)</h1>

In [1]:
# GIVEN — do not edit this cell (the grader uses its own copy)
import json, re, copy, time, datetime as dt
from types import SimpleNamespace

# The salon's clock is INJECTED. Never call datetime.now() in your assistant: use the `clock` you are given.
NOW = dt.datetime(2026, 10, 19, 10, 0)            # Monday 19 October 2026, 10:00

def fmt(t):      return t.strftime('%Y-%m-%d %H:%M')
def parse_dt(s): return dt.datetime.strptime(s, '%Y-%m-%d %H:%M')

SERVICES = {   # minutes are a multiple of 30; prices are INTEGER CENTS
    'cut':        {'name': 'Haircut',      'minutes': 30,  'price_cents': 3500},
    'beard':      {'name': 'Beard trim',   'minutes': 30,  'price_cents': 2000},
    'wash_style': {'name': 'Wash & style', 'minutes': 60,  'price_cents': 4500},
    'color':      {'name': 'Colour',       'minutes': 120, 'price_cents': 9000},
    'highlights': {'name': 'Highlights',   'minutes': 150, 'price_cents': 13000},
}
STYLISTS = {   # weekday(): Monday=0 ... Sunday=6
    'mara': {'name': 'Mara', 'services': {'cut', 'wash_style', 'color', 'highlights'}, 'days_off': {6}},
    'jon':  {'name': 'Jon',  'services': {'cut', 'beard', 'wash_style'},               'days_off': {2, 6}},
    'lea':  {'name': 'Lea',  'services': {'cut', 'color', 'highlights'},               'days_off': {0, 6}},
}
OPEN_H, CLOSE_H, SLOT_MIN, MAX_ACTIVE = 9, 18, 30, 3
SEED = [   # (customer, service, stylist, start)
    ('c_ada', 'cut',        'mara', '2026-10-20 10:00'),   # B-1  exactly 24h from NOW
    ('c_ada', 'color',      'lea',  '2026-10-23 14:00'),   # B-2
    ('c_ben', 'beard',      'jon',  '2026-10-19 15:00'),   # B-3  today, 5h away
    ('c_cy',  'cut',        'jon',  '2026-10-20 13:00'),   # B-4
    ('c_cy',  'wash_style', 'mara', '2026-10-22 11:00'),   # B-5
    ('c_cy',  'beard',      'jon',  '2026-10-24 16:00'),   # B-6  (c_cy has the maximum of 3 active bookings)
]

class BackendTimeout(Exception):
    """The salon system did not answer in time (the request may or may not have been applied for writes)."""
class SlotTaken(Exception):
    """Somebody else holds that slot."""
class BackendError(Exception):
    """An expected business refusal; .code is a short machine-readable reason."""
    def __init__(self, code):
        super().__init__(code); self.code = code

class SalonBackend:
    """The salon's booking system. READ it, do not edit it. Every public method is a possible failure point."""
    def __init__(self, now=NOW):
        self.now = now
        self.bookings, self.by_key, self.tickets = {}, {}, []
        self.next_id = 1
        self.call_log, self.write_log = [], []          # every method called / every COMMITTED write
        self.faults = {'fail_reads': 0, 'fail_next_write': 0, 'lose_next_write_response': 0}
        for customer, service, stylist, start in SEED:
            self._insert(customer, service, stylist, start)

    # ---------- helpers ----------
    def _log(self, name): self.call_log.append(name)
    def _read_fault(self):
        if self.faults['fail_reads'] > 0:
            self.faults['fail_reads'] -= 1
            raise BackendTimeout('read timeout')
    def _insert(self, customer, service, stylist, start):
        bid = f'B-{self.next_id}'; self.next_id += 1
        self.bookings[bid] = {'booking_id': bid, 'customer_id': customer, 'service_id': service, 'stylist_id': stylist,
                              'start': start, 'status': 'active', 'reschedules': 0, 'fee_cents': 0}
        return self.bookings[bid]
    def _busy(self, stylist, ignore=None):
        out = []
        for b in self.bookings.values():
            if b['status'] == 'active' and b['stylist_id'] == stylist and b['booking_id'] != ignore:
                s = parse_dt(b['start']); out.append((s, s + dt.timedelta(minutes=SERVICES[b['service_id']]['minutes'])))
        return out
    def _is_free(self, service, stylist, start, ignore=None):
        s = parse_dt(start); e = s + dt.timedelta(minutes=SERVICES[service]['minutes'])
        if s <= self.now or s.weekday() in STYLISTS[stylist]['days_off'] or service not in STYLISTS[stylist]['services']:
            return False
        if s.hour < OPEN_H or e > s.replace(hour=CLOSE_H, minute=0) or s.minute % SLOT_MIN:
            return False
        return all(e <= bs or s >= be for bs, be in self._busy(stylist, ignore))
    def _fee(self, booking):
        price = SERVICES[booking['service_id']]['price_cents']
        delta = parse_dt(booking['start']) - self.now
        if delta <= dt.timedelta(0): raise BackendError('already_started')
        if delta >= dt.timedelta(hours=24): return 0
        if delta >= dt.timedelta(hours=2): return price // 2
        return price
    def _replay(self, key, sig):
        entry = self.by_key.get(key)
        if entry is None: return None
        if entry['sig'] != sig: raise BackendError('key_reuse')
        return copy.deepcopy(entry['result'])
    def _write_faults(self):
        if self.faults['fail_next_write'] > 0:
            self.faults['fail_next_write'] -= 1
            raise BackendTimeout('write timeout (nothing was applied)')
    def _lost_response(self):
        if self.faults['lose_next_write_response'] > 0:
            self.faults['lose_next_write_response'] -= 1
            raise BackendTimeout('response lost (the write WAS applied)')

    # ---------- reads ----------
    def list_services(self):
        self._log('list_services'); self._read_fault(); return copy.deepcopy(SERVICES)
    def free_slots(self, service_id, date, stylist_id=None):
        self._log('free_slots'); self._read_fault()
        if service_id not in SERVICES: raise ValueError('unknown service')
        day = dt.datetime.strptime(date, '%Y-%m-%d')
        out = []
        for stylist in sorted(STYLISTS if stylist_id is None else [stylist_id]):
            t = day.replace(hour=OPEN_H)
            while t.hour < CLOSE_H:
                if self._is_free(service_id, stylist, fmt(t)):
                    out.append({'stylist_id': stylist, 'start': fmt(t)})
                t += dt.timedelta(minutes=SLOT_MIN)
        return sorted(out, key=lambda r: (r['start'], r['stylist_id']))
    def bookings_for(self, customer_id):
        self._log('bookings_for'); self._read_fault()
        return copy.deepcopy([b for b in self.bookings.values() if b['customer_id'] == customer_id])
    def find_by_key(self, request_key):
        """Look up the stored result of an earlier write. Never faults."""
        self._log('find_by_key')
        entry = self.by_key.get(request_key)
        return copy.deepcopy(entry['result']) if entry else None

    # ---------- writes (idempotent on request_key) ----------
    def book(self, customer_id, service_id, stylist_id, start, request_key):
        self._log('book'); self._write_faults()
        sig = ('book', customer_id, service_id, stylist_id, start)
        done = self._replay(request_key, sig)
        if done is None:
            if service_id not in SERVICES or stylist_id not in STYLISTS: raise BackendError('unknown_item')
            if sum(1 for b in self.bookings.values() if b['customer_id'] == customer_id and b['status'] == 'active') >= MAX_ACTIVE:
                raise BackendError('limit_reached')
            if not self._is_free(service_id, stylist_id, start): raise SlotTaken(start)
            done = copy.deepcopy(self._insert(customer_id, service_id, stylist_id, start))
            self.by_key[request_key] = {'sig': sig, 'result': done}
            self.write_log.append(('book', done['booking_id']))
        self._lost_response()
        return done
    def cancel(self, customer_id, booking_id, fee_cents, request_key):
        self._log('cancel'); self._write_faults()
        sig = ('cancel', customer_id, booking_id, fee_cents)
        done = self._replay(request_key, sig)
        if done is None:
            b = self.bookings.get(booking_id)
            if b is None or b['customer_id'] != customer_id: raise BackendError('not_found')
            if b['status'] != 'active': raise BackendError('already_cancelled')
            if fee_cents != self._fee(b): raise BackendError('fee_mismatch')
            b['status'], b['fee_cents'] = 'cancelled', fee_cents
            done = {'booking_id': booking_id, 'fee_cents': fee_cents, 'status': 'cancelled'}
            self.by_key[request_key] = {'sig': sig, 'result': done}
            self.write_log.append(('cancel', booking_id))
        self._lost_response()
        return copy.deepcopy(done)
    def reschedule(self, customer_id, booking_id, new_start, request_key):
        self._log('reschedule'); self._write_faults()
        sig = ('reschedule', customer_id, booking_id, new_start)
        done = self._replay(request_key, sig)
        if done is None:
            b = self.bookings.get(booking_id)
            if b is None or b['customer_id'] != customer_id: raise BackendError('not_found')
            if b['status'] != 'active': raise BackendError('already_cancelled')
            if parse_dt(b['start']) - self.now < dt.timedelta(hours=24): raise BackendError('too_late')
            if b['reschedules'] >= 1: raise BackendError('already_rescheduled')
            if not self._is_free(b['service_id'], b['stylist_id'], new_start, ignore=booking_id): raise SlotTaken(new_start)
            b['start'], b['reschedules'] = new_start, b['reschedules'] + 1
            done = copy.deepcopy(b)
            self.by_key[request_key] = {'sig': sig, 'result': done}
            self.write_log.append(('reschedule', booking_id))
        self._lost_response()
        return done
    def create_ticket(self, ticket):
        """Hand a case to a human. `ticket` must be a dict with customer_id, reason, summary."""
        self._log('create_ticket')
        ticket = {**ticket, 'ticket_id': f'T-{len(self.tickets) + 1}'}
        self.tickets.append(ticket)
        return ticket['ticket_id']

# every seed booking must be valid: build one backend and check the invariants once
_b = SalonBackend()
assert all(_b._is_free(b['service_id'], b['stylist_id'], b['start'], ignore=b['booking_id']) for b in _b.bookings.values()), 'seed data must be valid'
print('Salon backend ready:', len(_b.bookings), 'seed bookings. NOW =', fmt(NOW), '(Monday)')

Salon backend ready: 6 seed bookings. NOW = 2026-10-19 10:00 (Monday)


In [2]:
# GIVEN — do not edit this cell (the grader uses its own copy)
SAFE_FALLBACK = "I can't confirm that from the salon system. Let me check again, or I can connect you with a team member."

def call(cid, name, **arguments):      return {'id': cid, 'name': name, 'arguments': arguments}
def calls(*cs):                        return {'tool_calls': list(cs)}
def final(text):                       return {'final': text}

def check_transcript(messages):
    """Rules every model call must satisfy. Returns a list of violations (empty = a valid conversation)."""
    bad, open_ids, seen_ids = [], None, set()
    if not messages or messages[0].get('role') != 'system':
        bad.append('the first message must be the system prompt')
    for i, m in enumerate(messages):
        role = m.get('role')
        if role not in ('system', 'user', 'assistant', 'tool'):
            bad.append(f'message {i}: unknown role {role!r}'); continue
        if role == 'tool':
            if open_ids is None or m.get('tool_call_id') not in open_ids:
                bad.append(f'message {i}: tool message {m.get("tool_call_id")!r} does not answer a pending tool call')
            else:
                open_ids.discard(m['tool_call_id'])
            try: json.loads(m.get('content'))
            except Exception: bad.append(f'message {i}: tool content must be a JSON string')
            continue
        if open_ids:
            bad.append(f'message {i}: tool calls {sorted(open_ids)} were never answered')
        open_ids = None
        if role == 'assistant' and m.get('tool_calls'):
            ids = [c.get('id') for c in m['tool_calls']]
            if len(set(ids)) != len(ids) or seen_ids & set(ids):
                bad.append(f'message {i}: duplicate tool call id')
            seen_ids |= set(ids); open_ids = set(ids)
    if open_ids:
        bad.append(f'tool calls {sorted(open_ids)} were never answered')
    if len(messages) > 24:
        bad.append(f'{len(messages)} messages sent to the model (limit 24)')
    return bad

class TurnScriptedModel:
    """A scripted 'model'. Each scenario turn loads the steps the model will take; every model call consumes one step."""
    def __init__(self):
        self.queue, self.violations, self.max_len, self.n_calls = [], [], 0, 0
        self.tools_seen, self.snapshots, self.turn_label = None, [], ''
    def load(self, steps, label=''):
        self.queue, self.turn_label = list(steps), label
    def __call__(self, messages, tools=None):
        self.n_calls += 1
        self.tools_seen = tools
        self.max_len = max(self.max_len, len(messages))
        self.snapshots.append(copy.deepcopy(messages))
        for v in check_transcript(messages):
            self.violations.append(f'[{self.turn_label}] {v}')
        if not self.queue:
            self.violations.append(f'[{self.turn_label}] the assistant called the model more often than this scenario allows')
            return {'final': 'SCRIPT-EXHAUSTED'}
        step = self.queue.pop(0)
        return copy.deepcopy(step(messages) if callable(step) else step)
    def observations(self):
        """Every tool observation the model has been shown so far (parsed, in order of first appearance)."""
        seen, out = set(), []
        for snap in self.snapshots:
            for m in snap:
                if m.get('role') == 'tool' and m['tool_call_id'] not in seen:
                    seen.add(m['tool_call_id'])
                    try: out.append({**json.loads(m['content']), '_id': m['tool_call_id']})
                    except Exception: pass
        return out

def T(user, *steps, session='s1', before=None, after=None, has=(), lacks=(), equals=None):
    return dict(user=user, model=list(steps), session=session, before=before, after=after,
                has=list(has), lacks=list(lacks), equals=equals)

def active(backend, customer=None):
    return [b for b in backend.bookings.values() if b['status'] == 'active' and (customer is None or b['customer_id'] == customer)]
def new_bookings(backend):
    return [b for b in backend.bookings.values() if int(b['booking_id'][2:]) > len(SEED)]
def tool_events(assistant):
    return [e for e in assistant.audit if e.get('tool') != '(user)']

def run_scenario(sc, assistant_cls, trace=False):
    """Run one scenario against a fresh salon. Returns None (pass) or a short failure message."""
    backend = SalonBackend()
    if sc.get('setup'): sc['setup'](backend)
    model = TurnScriptedModel()
    try:
        assistant = assistant_cls(model, backend, lambda: backend.now)
        sessions = sc.get('sessions') or {'s1': sc.get('customer', 'c_ada')}
        for sid, cust in sessions.items():
            assistant.open_session(sid, cust)
        ctx = SimpleNamespace(backend=backend, assistant=assistant, model=model, replies=[], sessions=sessions)
        for n, turn in enumerate(sc.get('turns', []), 1):
            label = f'turn {n} ({turn["user"][:28]!r})'
            if turn['before']: turn['before'](backend)
            model.load(turn['model'], label)
            reply = assistant.handle(turn['session'], turn['user'])
            ctx.replies.append(reply)
            if trace:
                print(f'\n[{label}]\n  USER : {turn["user"]}\n  REPLY: {reply}')
            if not isinstance(reply, str): return f'{label}: handle() must return a string'
            if model.queue and not sc.get('loose'): return f'{label}: the assistant called the model fewer times than expected ({len(model.queue)} script steps unused)'
            for needle in turn['has']:
                if needle.lower() not in reply.lower(): return f'{label}: reply should contain {needle!r}; got {reply[:90]!r}'
            for needle in turn['lacks']:
                if needle.lower() in reply.lower(): return f'{label}: reply must not contain {needle!r}'
            if turn['equals'] is not None and reply != turn['equals']: return f'{label}: expected exactly {turn["equals"][:50]!r}; got {reply[:70]!r}'
            if turn['after']:
                try: turn['after'](ctx)
                except AssertionError as exc: return f'{label}: {exc}'
        if sc.get('check'):
            sc['check'](ctx)
        if model.violations: return model.violations[0]
        return None
    except AssertionError as exc:
        return f'{exc}' or 'assertion failed'
    except NotImplementedError as exc:
        return f'not implemented yet ({exc})'
    except Exception as exc:
        return f'{type(exc).__name__}: {str(exc)[:110]}'

def run_suite(scenarios, assistant_cls, title='Public suite', total_points=None, verbose=True):
    results, earned, wsum = [], 0.0, sum(s['weight'] for s in scenarios)
    for sc in scenarios:
        err = run_scenario(sc, assistant_cls)
        results.append((sc, err))
        pts = (total_points or wsum) * sc['weight'] / wsum
        earned += 0 if err else pts
        if verbose:
            print(('✓' if err is None else '✗'), f"{sc['id']} {sc['title']}", '' if err is None else f'→ {err}')
    passed = sum(1 for _, e in results if e is None)
    print(f'{title}: {passed}/{len(scenarios)} scenarios passed' + (f'  →  {earned:.1f}/{total_points} points' if total_points else ''))
    return earned

def trace(scenario_id, assistant_cls, scenarios=None):
    """Debug helper: run ONE scenario and print every user message and reply, the failure (if any), and the audit trail."""
    pool = scenarios if scenarios is not None else PUBLIC
    sc = next(s for s in pool if s['id'] == scenario_id)
    print(f"=== {sc['id']} {sc['title']} ===")
    err = run_scenario(sc, assistant_cls, trace=True)
    print('\nRESULT:', 'PASS' if err is None else f'FAIL → {err}')

In [3]:
# GIVEN — do not edit this cell (the grader uses its own copy)
def obs_by_id(ctx):
    return {o['_id']: o for o in ctx.model.observations()}

def _nothing_written(ctx):
    assert ctx.backend.write_log == [], 'a proposal must not change anything'
    assert new_bookings(ctx.backend) == []

def _propose_cut(cid='b1'):
    return T('Please book me a haircut with Mara tomorrow at 11:00',
             calls(call(cid, 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-20 11:00')),
             final('Here is what I will book: Haircut with Mara on 2026-10-20 at 11:00 for €35.00. Shall I go ahead?'),
             has=['Shall I'], after=_nothing_written)

def sc_greeting(ctx):
    assert ctx.backend.call_log == [], f'a greeting needs no backend call, but you called {ctx.backend.call_log}'
    assert tool_events(ctx.assistant) == [] and ctx.model.n_calls == 1

def sc_browse(ctx):
    o = obs_by_id(ctx)
    assert o['a1']['status'] == 'ok' and o['a2']['status'] == 'ok', o
    assert [m['tool_call_id'] for m in ctx.model.snapshots[-1] if m['role'] == 'tool'] == ['a1', 'a2']
    assert ctx.backend.write_log == []

def sc_book_happy(ctx):
    mine = new_bookings(ctx.backend)
    assert len(mine) == 1 and ctx.backend.write_log == [('book', 'B-7')], ctx.backend.write_log
    b = mine[0]
    assert (b['customer_id'], b['service_id'], b['stylist_id'], b['start']) == ('c_ada', 'cut', 'mara', '2026-10-20 11:00')
    assert 'Now: 2026-10-19 10:00' in ctx.model.snapshots[0][0]['content'], 'the system prompt must contain the line "Now: YYYY-MM-DD HH:MM" from the clock'

def sc_unclear_and_bare_yes(ctx):
    assert ctx.backend.write_log == [] and new_bookings(ctx.backend) == []

def sc_race(ctx):
    assert ctx.backend.write_log == [('book', 'B-7')], 'only the rival booking may exist'
    last = [o for o in ctx.model.observations() if o.get('tool') == 'book_appointment' and o.get('status') == 'error'][-1]
    assert last['error_type'] == 'slot_taken', last
    alts = last['data']['alternatives']
    assert alts[:2] == [{'stylist_id': 'jon', 'start': '2026-10-20 11:00'}, {'stylist_id': 'lea', 'start': '2026-10-20 11:00'}], alts
    assert len(alts) == 3, 'offer up to 3 alternatives, nearest in time first'

def sc_cancel_free(ctx):
    assert ctx.backend.bookings['B-2']['status'] == 'cancelled' and ctx.backend.bookings['B-2']['fee_cents'] == 0
    assert ctx.backend.write_log == [('cancel', 'B-2')]

def sc_cancel_fee(ctx):
    b = ctx.backend.bookings['B-3']
    assert b['status'] == 'cancelled' and b['fee_cents'] == 1000, b

def sc_reschedule(ctx):
    b = ctx.backend.bookings['B-1']
    assert b['start'] == '2026-10-20 14:00' and b['reschedules'] == 1 and ctx.backend.write_log == [('reschedule', 'B-1')], (b, ctx.backend.write_log)
    last = [o for o in ctx.model.observations() if o.get('tool') == 'reschedule_appointment'][-1]
    assert last['status'] == 'blocked' and last['error_type'] == 'already_rescheduled', last

def sc_too_late(ctx):
    last = [o for o in ctx.model.observations() if o.get('tool') == 'reschedule_appointment'][-1]
    assert last['status'] == 'blocked' and last['error_type'] == 'too_late', last
    assert ctx.backend.write_log == []

def sc_identity(ctx):
    assert ctx.backend.bookings['B-3']['status'] == 'active'
    o = obs_by_id(ctx)
    assert o['i1']['status'] == 'not_found' and o['i2']['error_type'] == 'validation_error' and o['i3']['status'] == 'ok'
    assert 'B-3' not in json.dumps(o['i3']), "another customer's booking must never be shown"

def sc_limit(ctx):
    last = [o for o in ctx.model.observations() if o.get('tool') == 'book_appointment'][-1]
    assert last['status'] == 'blocked' and last['error_type'] == 'limit_reached', last
    assert ctx.backend.write_log == [] and 'book' not in ctx.backend.call_log

def sc_lost_response(ctx):
    assert len(new_bookings(ctx.backend)) == 1 and ctx.backend.write_log == [('book', 'B-7')], 'exactly one booking, even though the reply was lost'
    last = [o for o in ctx.model.observations() if o.get('tool') == 'book_appointment'][-1]
    assert last['status'] == 'ok' and last['data']['booking_id'] == 'B-7', last

def sc_read_retry(ctx):
    ev = [e for e in tool_events(ctx.assistant) if e['tool'] == 'find_slots']
    assert len(ev) == 1 and ev[0]['status'] == 'ok' and ev[0]['attempts'] == 2, ev

def sc_read_exhausted(ctx):
    assert len(ctx.backend.tickets) == 1 and ctx.backend.tickets[0]['reason'] == 'tool_failures', ctx.backend.tickets
    assert all(o['error_type'] == 'timeout' for o in ctx.model.observations()), ctx.model.observations()

def sc_grounding(ctx):
    assert ctx.replies[0] == SAFE_FALLBACK and ctx.replies[1] == SAFE_FALLBACK, ctx.replies
    assert ctx.replies[2].startswith('Yes: B-1'), ctx.replies[2]

def sc_bad_calls(ctx):
    o = obs_by_id(ctx)
    assert [(o[i]['status'], o[i].get('error_type')) for i in ('x1', 'x2', 'x3', 'x4')] == \
        [('error', 'validation_error'), ('error', 'unknown_tool'), ('error', 'validation_error'), ('error', 'validation_error')], o
    assert ctx.backend.write_log == [] and 'book' not in ctx.backend.call_log

def sc_long_chat(ctx):
    assert len(new_bookings(ctx.backend)) == 1, 'the booking made after 30 chatty turns must still work'
    assert ctx.model.max_len <= 24, f'{ctx.model.max_len} messages sent to the model'

def sc_multi_intent(ctx):
    assert ctx.backend.bookings['B-2']['status'] == 'cancelled' and new_bookings(ctx.backend) == []
    o = obs_by_id(ctx)
    assert o['m1']['status'] == 'needs_confirmation' and o['m2']['status'] == 'blocked' and o['m2']['error_type'] == 'pending_exists', o

def sc_handoff(ctx):
    assert len(ctx.backend.tickets) == 1
    t = ctx.backend.tickets[0]
    assert t['reason'] == 'customer_request' and t['customer_id'] == 'c_ada' and isinstance(t['summary'], str) and len(t['summary']) <= 300, t
    blob = json.dumps(ctx.backend.tickets) + json.dumps(ctx.assistant.audit)
    assert '@' not in blob and '1234567' not in blob, 'tickets and audit logs must not contain the e-mail address or phone number'

def sc_schemas(ctx):
    tools = ctx.model.tools_seen
    assert isinstance(tools, list) and len(tools) == 7, 'pass your 7 tool schemas as the second argument of model(...)'
    names = {t['function']['name'] for t in tools}
    assert names == {'list_services', 'find_slots', 'get_my_bookings', 'book_appointment', 'reschedule_appointment', 'cancel_appointment', 'handoff_to_human'}, names
    for t in tools:
        p = t['function']['parameters']
        assert t['type'] == 'function' and p['type'] == 'object' and p['additionalProperties'] is False, t['function']['name']
        assert not ({'customer_id', 'request_key', 'approved', 'price', 'fee_cents'} & set(p['properties'])), f"{t['function']['name']}: the model must not control identity, keys, prices or approvals"
        assert set(p.get('required', [])) <= set(p['properties'])
    assert set(next(t for t in tools if t['function']['name'] == 'book_appointment')['function']['parameters']['required']) == {'service_id', 'stylist_id', 'start'}

def sc_fee_table(ctx):
    cases = [(3500, '2026-10-20 10:00', 0), (3500, '2026-10-20 09:59', 1750), (3500, '2026-10-19 12:00', 1750), (3500, '2026-10-19 11:59', 3500),
             (2000, '2026-10-19 15:00', 1000), (9000, '2026-10-23 14:00', 0), (4500, '2026-10-19 11:30', 4500), (2001, '2026-10-19 20:00', 1000)]
    for price, start, want in cases:
        got = cancellation_fee_cents(price, parse_dt(start), NOW)
        assert got == want, f'fee({price}, {start}) = {got}, expected {want}'
    try:
        cancellation_fee_cents(3500, parse_dt('2026-10-19 10:00'), NOW)
    except ValueError:
        return
    raise AssertionError('a booking that has already started cannot be cancelled: raise ValueError')

REPLY_CASES = [('yes','yes'),('Yes, please!','yes'),('YES','yes'),('ok','yes'),('okay thanks','yes'),('sure','yes'),('y','yes'),('go ahead','yes'),
               ('that works','yes'),('sounds good','yes'),('perfect','yes'),('yes!!!','yes'),('Yes please do','yes'),('absolutely','yes'),
               ('no','no'),('No thanks','no'),('nope','no'),('nah','no'),("don't book it",'no'),('not yet','no'),('never mind','no'),('stop','no'),
               ('yes but 3pm instead','unclear'),('yes and also add a beard trim','unclear'),('maybe','unclear'),('?','unclear'),('what does it cost?','unclear'),
               ('no, yes, no','unclear'),('yes 15:00','unclear'),('cancel my friday appointment','unclear'),('I would like that but later','unclear'),
               ('','unclear'),('   ','unclear'),('ok but wait','unclear'),('yes, and can you also move B-2?','unclear'),('Can you do 4pm?','unclear')]

def sc_reply_table(ctx):
    wrong = [(t, want, interpret_reply(t)) for t, want in REPLY_CASES if interpret_reply(t) != want]
    assert not wrong, f'{len(wrong)} wrong, e.g. interpret_reply({wrong[0][0]!r}) = {wrong[0][2]!r}, expected {wrong[0][1]!r}'

def _fillers(n): return [T('ok thanks', final("You're welcome!")) for _ in range(n)]

PUBLIC = [
 dict(id='P01', title='Greeting needs no tools', weight=1, turns=[T('Hi!', final('Hello! How can I help you today?'), has=['Hello'])], check=sc_greeting),
 dict(id='P02', title='Browse: two independent reads in one turn', weight=2, turns=[T("What do you offer, and what's free for a haircut on Tuesday?",
      calls(call('a1', 'list_services'), call('a2', 'find_slots', service_id='cut', date='2026-10-20')),
      final('We offer Haircut (€35.00), Beard trim, Wash & style, Colour and Highlights. Tuesday haircut slots start at 09:00 and 09:30.'), has=['09:00', '€35'])], check=sc_browse),
 dict(id='P03', title='Book: propose, then confirm in a LATER turn', weight=3, turns=[_propose_cut(),
      T('yes please', final('Done! Your haircut with Mara is booked for 2026-10-20 at 11:00 (B-7).'), has=['B-7'])],
      check=sc_book_happy),
 dict(id='P04', title='"yes, but..." and a bare "yes" never book', weight=3, turns=[_propose_cut(),
      T('yes but make it 15:00 instead', final('Sure, let me look at 15:00 instead.')),
      T('yes', final('Which appointment do you mean?'))], check=sc_unclear_and_bare_yes),
 dict(id='P05', title='interpret_reply: 36 customer replies', weight=3, check=sc_reply_table),
 dict(id='P06', title='Race: the slot is taken between proposal and "yes"', weight=3, turns=[_propose_cut(),
      T('yes', final("I'm sorry, 11:00 was just taken. I can offer 11:00 with Jon or Lea, or 10:30 with Jon."), has=['11:00'],
        before=lambda b: b.book('c_ben', 'cut', 'mara', '2026-10-20 11:00', 'rival-1'))], check=sc_race),
 dict(id='P07', title='Cancel more than 24h ahead is free', weight=2, turns=[
      T('I need to cancel my Friday colour appointment', calls(call('c1', 'get_my_bookings')),
        calls(call('c2', 'cancel_appointment', booking_id='B-2')),
        final('Cancelling B-2 (Colour on 2026-10-23 at 14:00) is free of charge. Shall I cancel it?')),
      T('yes', final('B-2 is cancelled. No fee was charged.'), has=['cancelled'])], check=sc_cancel_free),
 dict(id='P08', title='Late cancellation: fee shown BEFORE the customer says yes', weight=3, customer='c_ben', turns=[
      T('Cancel my appointment today', calls(call('c1', 'cancel_appointment', booking_id='B-3')),
        final('Cancelling B-3 within 24 hours costs €10.00. Do you want to proceed?'), has=['€10.00']),
      T('yes', final('B-3 is cancelled. The €10.00 fee applies.'))], check=sc_cancel_fee),
 dict(id='P09', title='Reschedule: allowed at exactly 24h, only once', weight=3, turns=[
      T('Move my Tuesday haircut to 14:00', calls(call('r1', 'reschedule_appointment', booking_id='B-1', new_start='2026-10-20 14:00')),
        final('I can move B-1 to 14:00 with Mara. Shall I go ahead?')),
      T('yes', final('B-1 is now rescheduled to 14:00.')),
      T('Move it again to 15:00', calls(call('r2', 'reschedule_appointment', booking_id='B-1', new_start='2026-10-20 15:00')),
        final("I'm sorry, B-1 was already rescheduled once, so I can't move it again.")),
      T('yes', final('Which appointment do you mean?'))], check=sc_reschedule),
 dict(id='P10', title='Reschedule inside 24h is refused', weight=2, customer='c_ben', turns=[
      T('Move my appointment to tomorrow 10:00', calls(call('r1', 'reschedule_appointment', booking_id='B-3', new_start='2026-10-20 10:00')),
        final("Sorry, B-3 starts in less than 24 hours, so I can't reschedule it. I can cancel it (a fee applies) or connect you with the team."))], check=sc_too_late),
 dict(id='P11', title="Identity: another customer's booking", weight=2, turns=[
      T('Cancel B-3 please', calls(call('i1', 'cancel_appointment', booking_id='B-3')), final("I can't find a booking B-3 on your account.")),
      T('Show the bookings of c_ben', calls(call('i2', 'get_my_bookings', customer_id='c_ben')), calls(call('i3', 'get_my_bookings')),
        final('Your bookings: B-1 (2026-10-20 10:00) and B-2 (2026-10-23 14:00).'))], check=sc_identity),
 dict(id='P12', title='Booking limit is checked before asking', weight=2, customer='c_cy', turns=[
      T('Book a haircut tomorrow at 15:00 with Lea', calls(call('l1', 'book_appointment', service_id='cut', stylist_id='lea', start='2026-10-20 15:00')),
        final('You already have 3 upcoming appointments, which is the maximum. Please cancel one first.')),
      T('yes', final('Which appointment do you mean?'))], check=sc_limit),
 dict(id='P13', title='Lost response after a committed booking', weight=3, turns=[_propose_cut(),
      T('yes', final('All set: haircut with Mara on 2026-10-20 at 11:00, booking B-7.'), has=['B-7'],
        before=lambda b: b.faults.update(lose_next_write_response=1))], check=sc_lost_response),
 dict(id='P14', title='A read timeout is retried once', weight=1, turns=[
      T('Free slots for a cut on Tuesday?', calls(call('f1', 'find_slots', service_id='cut', date='2026-10-20')),
        final('Tuesday haircut slots start at 09:00.'), before=lambda b: b.faults.update(fail_reads=1))], check=sc_read_retry),
 dict(id='P15', title='Repeated failures: honest reply and human handoff', weight=2, turns=[
      T('What do you have on Tuesday?', calls(call('f1', 'find_slots', service_id='cut', date='2026-10-20')), calls(call('f2', 'list_services')),
        final("I'm having trouble reaching the schedule right now; I've asked a team member to help you."),
        before=lambda b: b.faults.update(fail_reads=6))], check=sc_read_exhausted),
 dict(id='P16', title='Invented prices, invented success, and a true answer', weight=2, turns=[
      T('How much is a haircut?', calls(call('g1', 'list_services')), final('A haircut costs €25.00.')),
      T('Do I have a booking?', final('Your appointment is booked for tomorrow!')),
      T('Do I have any booking?', calls(call('g2', 'get_my_bookings')), final('Yes: B-1 on 2026-10-20 at 10:00.'))], check=sc_grounding),
 dict(id='P17', title='Malformed, unknown and smuggling tool calls', weight=2, turns=[
      T('Book me something', calls(call('x1', 'book_appointment', service_id='cut', stylist_id='mara', start='tomorrow 11am'),
                                   call('x2', 'delete_all_bookings'),
                                   call('x3', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-20 11:00', customer_id='c_ben'),
                                   call('x4', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-20 11:15')),
        final('I need an exact time such as 2026-10-20 11:00. Which time would you like?'))], check=sc_bad_calls),
 dict(id='P18', title='30 chatty turns, then a booking (context window)', weight=2,
      turns=_fillers(30) + [_propose_cut('lb1'), T('yes', final('Booked: haircut with Mara on 2026-10-20 at 11:00 (B-7).'), has=['B-7'])], check=sc_long_chat),
 dict(id='P19', title='Two requests in one message: one pending action', weight=2, turns=[
      T('Cancel B-2 and book a cut on Saturday at 11:00 with Mara',
        calls(call('m1', 'cancel_appointment', booking_id='B-2'), call('m2', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-24 11:00')),
        final("Let's start with cancelling B-2 (free of charge). Shall I cancel it? Then we'll book Saturday.")),
      T('yes', final('B-2 is cancelled.'))], check=sc_multi_intent),
 dict(id='P20', title='Human handoff without leaking personal data', weight=2, turns=[
      T('I want a person. My email is ada@example.com and my phone is +49 170 1234567',
        calls(call('h1', 'handoff_to_human', reason='customer_request')), final("I've passed your request to our team; someone will contact you."))], check=sc_handoff),
 dict(id='P21', title='Tool schemas', weight=1, turns=[T('Hi', final('Hello!'))], check=sc_schemas),
 dict(id='P22', title='cancellation_fee_cents: boundaries', weight=2, check=sc_fee_table),
]

def run_public(assistant_cls, **kw):
    return run_suite(PUBLIC, assistant_cls, 'Public suite', total_points=40, **kw)
print(len(PUBLIC), 'public scenarios loaded.')

22 public scenarios loaded.
